# Script creating new .inp files for Loadest

Input: 
1. Original csv file from the NWT community database (spring 2019)
2. Hydrometric flow data (m3/s) until 2016

Output: 
1. new calib.inp files for all NWT variables

Comments:

Only includes sample instances where Hg was collected

A.L.Soerensen June 2019

Updates 2019-08-28
Change directories to staffans mac desktop.
Changed dates for startdate (2012-01-01) and enddate (2019-01-01) for the NWT.csv folder


In [43]:
#Libraries
import pandas as pd
import numpy as np
from datetime import datetime
from datetime import timedelta

######################################
# Data that might need to be modified
######################################
#
# Start date & End date
startdate = '2012-01-01 00:00:00'
enddate = '2019-01-01 00:00:00'

######################################
# Rivers to isolate
Rivers=['Fort McPherson at Peel River','Tsiigehtchic at Arctic Red River','Fort Simpson at Mackenzie River / Upstream of Liard River Mouth',
        'Yellowknife at Yellowknife River / Up Stream from Bridge']
# Short name of rivers used in .inp file (needs to match rivers above)
River_short=['Peel River','Arctic Red River','Liard River','Yellowknife River']

# Station to isolate for hydrological information
#            Peel       Liard    Arctic Red  Yellow
file_name = ['10MC002','10ED002','10LA002','07SB003']

########################################
# Constituent lists (names and unit)
#
# List of constituent to make output files for
NWT_list = ['THg_ugL','DHg_ugL','TOC_mgL','DOC_mgL','TS_mgL','TSS_mgL','TDS_mgL']

################################################
# File paths
#
# path for NWT monitoring data
NWT_name='/Users/staffan/Dropbox/Mackenzie Hg project/Supporting data/NWT Community-based water quality monitoring program'

# Path for hydrometric water flow file
Flow_name = '/Users/staffan/Dropbox/Mackenzie Hg project/Supporting data/Hydrological_data/Daily_'

# Base path to Loadest folder
Loadest = '/Users/staffan/Desktop/Loadest/'



In [44]:
######################################################
# No need to change
#
# Path for original .inp file used as a template for all the new files
path = Loadest+'/calib.inp'

# Paths for temporary files
Temp_name = Loadest+'/txt_temp'

# Paths for final output files
Out_name = Loadest+'/new_files'

In [45]:
###################################
# read NWT monitoring data file
#
fname = NWT_name + '/NWT-wide Community-based Monitoring Program_data.csv'
df = pd.read_csv(fname, sep=',', low_memory=False)
dft = df.set_index('LaboratoryName').drop('Field Data',axis=0)
df = dft.reset_index()
df.head()

,LaboratoryName,DatasetName,MonitoringLocationName,MonitoringLocationID,MonitoringLocationLatitude,MonitoringLocationLongitude,MonitoringLocationHorizontalCoordinateReferenceSystem,MonitoringLocationType,MonitoringLocationWaterbody,ActivityType,...,ResultDetectionQuantitationLimitType,ResultStatusID,ResultComment,ResultAnalyticalMethodID,ResultAnalyticalMethodContext,ResultAnalyticalMethodName,AnalysisStartDate,AnalysisStartTime,AnalysisStartTimeZone,LaboratorySampleID
0,Taiga,NWT-wide Community-based Monitoring Program,Aklavik at Mackenzie River Delta / Peel Channel,NaN,68.139121,-135.040791,WGS84,River/Stream,NaN,Sample-Routine,...,NaN,NaN,CBM-2015-00017-002,NaN,NaN,NaN,2015-08-17,09:00:00,-600.0,NaN
1,Taiga,NWT-wide Community-based Monitoring Program,Tulita at Mackenzie River,NaN,64.891618,-125.531380,WGS84,River/Stream,NaN,Sample-Routine,...,NaN,NaN,CBM-2016-00026-006,NaN,NaN,NaN,2016-08-05,09:00:00,-600.0,NaN
2,Taiga,NWT-wide Community-based Monitoring Program,Tulita at Great Bear River,NaN,64.998914,-125.296917,WGS84,River/Stream,NaN,Sample-Routine,...,NaN,NaN,CBM-2016-00026-005,NaN,NaN,NaN,2016-08-05,09:00:00,-600.0,NaN
3,Taiga,NWT-wide Community-based Monitoring Program,Fort Good Hope at Rabbit Skin River,NaN,66.295000,-128.605750,WGS84,River/Stream,NaN,Sample-Routine,...,NaN,NaN,CBM-2015-00015-015,NaN,NaN,NaN,2015-08-12,09:00:00,-600.0,NaN
4,Taiga,NWT-wide Community-based Monitoring Program,Norman Wells at Bosworth Creek / Upstream of C...,NaN,65.283717,-126.864850,WGS84,River/Stream,NaN,Sample-Routine,...,NaN,NaN,CBM-2017-00026-001,NaN,NaN,NaN,2017-08-25,09:00:00,-600.0,NaN


In [46]:
df = df.drop(['DatasetName','MonitoringLocationID','MonitoringLocationWaterbody','ActivityEndDate','ActivityEndTime',
             'ResultDetectionQuantitationLimitType','ResultStatusID','ResultAnalyticalMethodID','ResultAnalyticalMethodContext',
             'ResultAnalyticalMethodName','AnalysisStartDate','AnalysisStartTime','AnalysisStartTimeZone',
             'LaboratoryName','LaboratorySampleID','MethodSpeciation','ActivityStartTime',
             'MonitoringLocationHorizontalCoordinateReferenceSystem','ActivityType','ActivityMediaName',
             'SampleCollectionEquipmentName'],axis=1)

df=df.rename(index=str, columns={"MonitoringLocationName":"Location","MonitoringLocationLatitude":"Latitude",
                                "MonitoringLocationLongitude":"Longitude","MonitoringLocationType":"Waterbody type",
                                "CharacteristicName":"Sample_type","ActivityStartDate":"Date",
                                 "ResultDetectionCondition":"Below_detection","ResultDetectionQuantitationLimitMeasure":"LOD",
                                "ActivityDepthHeightMeasure":"Depth","ActivityDepthHeightUnit":"Depth_unit",
                                "ResultDetectionQuantitationLimitUnit":"LOD_unit"})

# Convert date to Datetime
df["Date"]=pd.to_datetime(df.Date)

# Fill nan for Time column
values = {'ResultComment': 'None'}
df = df.fillna(value=values)

df['Depth'] = df['Depth']*(-1)
values = {'Depth': 0, 'Depth_unit': 'm'}
df = df.fillna(value=values)

# Extract only River/stream water bodies, not Lake and ponds
df1 = df.set_index('Waterbody type')
df1 = df1.loc['River/Stream'] 
df1 = df1.reset_index().drop('Waterbody type', axis=1)

# Drop unwanted elements
df1 = df1.set_index('Sample_type').drop(['Antimony','Bismuth','Beryllium','Boron','Cesium','Cobalt','Chromium','Molybdenum','Orthophosphate',
                                        'Uranium','True color','Silica, reactive','Silver','Chlorophyll a (probe)','Fecal Coliform','Selenium',
                                        'Ethylbenzene','Streptococcus','Toluene','Total Coliform','Escherichia coli','Enterococcus','Thallium',
                                        'Benzene','Oxidation reduction potential (ORP)','Dissolved oxygen (DO)    ','Temperature, water ',
                                        'Nitrite','Nitrate','Xylene','Tin','Hydrocarbons','Inorganic nitrogen (nitrate and nitrite)',
                                        'Dissolved oxygen saturation','Alkalinity, total','Aluminum','Ammonia','Arsenic','Barium','Cadmium',
                                        'Calcium','Chloride','Chlorophyll a','Copper','Fluoride','Hardness, carbonate','Iron','Lead','Lithium',
                                        'Magnesium','Manganese','Zinc','Nickel','Potassium','Vanadium','Nitrogen','Titanium','Strontium',
                                        'Sodium','Rubidium','Phosphorus','pH','Specific conductance','Turbidity'], axis=0)
df1 = df1.reset_index()

# Create unique names for variables - split and concat again
df1notnan = df1.dropna(subset=['ResultSampleFraction']).copy(deep=True)
df1notnan['Sample_type'] = df1notnan['Sample_type'] + df1notnan['ResultSampleFraction'] #A value is trying to be set on a copy of a slice from a DataFrame.

df1isnull = df1[pd.isnull(df1['ResultSampleFraction'])]

df1a=pd.concat([df1notnan,df1isnull]).drop('ResultSampleFraction',axis=1)#, sort=True)

#########################################################
# Remove non-trace metal Hg samples (ug/L)
df1b = df1a.set_index(['Sample_type'])
df1b_m = df1b.loc[['MercuryTotal','MercuryDissolved'],:]
df1b_m = df1b_m.reset_index()

df1b_m = df1b_m.set_index(['Sample_type','ResultUnit'])
df1b_m = df1b_m.drop('ug/L',level=1)
df1b_m = df1b_m.reset_index()

df1b_m = df1b_m.set_index(['Sample_type','LOD_unit'])
df1b_m = df1b_m.drop('ug/L',level=1)
df1b_m = df1b_m.reset_index()

df1b_r = df1b.drop(['MercuryTotal','MercuryDissolved'], axis=0)
df1b_r = df1b_r.reset_index()

df1c=pd.concat([df1b_m,df1b_r], sort=True).drop('LOD_unit',axis=1)

In [47]:
# make sure each variable only have one unit represented
#df_doub = df1c.set_index(['Sample_type','ResultUnit'])
#df_doub = df_doub.groupby(level = ['Sample_type','ResultUnit']).mean()
#df_doub

In [56]:
######################################################
# Create dataset with unique Location, dates and depths for where mercury data present
# used as basis for the dataset found in our shared folder
######################################################

df1d = df1c
df_loc0 = df1d.drop(['Depth_unit','ResultUnit','ResultValue','Below_detection','LOD'],axis=1)#'Sample_type',
df_loc0 = df_loc0.set_index('Sample_type').loc[['MercuryTotal','MercuryDissolved'],:]
df_loc0 = df_loc0.reset_index().drop('Sample_type',axis=1)
df_loc0 = df_loc0.drop_duplicates(subset=None, keep='first')
df_stat = df_loc0
df_loc  = df_loc0

# create lists with variable names in original spreadsheet and new names for new spreadsheet (that includes units)
df_variable=df1d.drop(['Location','Latitude','Longitude','Date','Depth','Depth_unit','ResultValue','Below_detection','LOD'],axis=1)
df_variable = df_variable.drop_duplicates(subset=['Sample_type'])
df_variable['Name']=df_variable['Sample_type']+' ('+df_variable['ResultUnit']+')'
df_variable['Name']=df_variable['Name'].fillna(df_variable['Sample_type'])

List=df_variable['Sample_type'].tolist()
List_name=df_variable['Name'].tolist()

# loop over all variables and merge with unique set of locations, depths and times
df2=df1d.set_index(['Sample_type']).drop(['Latitude','Longitude','Depth_unit','ResultUnit'],axis=1)
n=len(List)

# Treat missing values and <LOD samples
for id in range(n):
    Alk_t = df2.loc[[List[id]],:]
    Alk_t=Alk_t.rename(index=str, columns={"ResultValue":List_name[id]})
    Alk_t = Alk_t.reset_index().drop('Sample_type',axis=1)

    df2notnan = Alk_t.dropna(subset=[List_name[id]])
    df2isnull = Alk_t[pd.isnull(Alk_t[List_name[id]])].copy(deep=True)
    
    # How to treat <LOD data.... pick option
    #df2isnull[List_name[id]] = '<'+df2isnull['LOD'].astype(str) #A value is trying to be set on a copy of a slice from a DataFrame.
    df2isnull[List_name[id]] = df2isnull['LOD']*0.5                                  # Set <LOD to 0.5*LOD
    
    Alk_t1=pd.concat([df2notnan,df2isnull]).drop(['LOD','Below_detection'],axis=1)
        
    df_loc=pd.merge(df_loc,Alk_t1,how='left',left_on=['Location','Date','Depth','ResultComment'],right_on=['Location','Date','Depth','ResultComment'])

# Extract just data on River samples
df_loc = df_loc.set_index("Location")
df_loc=df_loc.loc[Rivers,:]
df_loc= df_loc.reset_index()

# Add river names based on ID
for id in range(len(Rivers)):
    df_loc.loc[df_loc.Location == Rivers[id], 'Location'] = River_short[id]

# Reformat Hg to correct units
df_loc['THg_ugL'] = df_loc['MercuryTotal (ng/L)']/1000
df_loc['DHg_ugL'] = df_loc['MercuryDissolved (ng/L)']/1000
df_loc = df_loc.rename(index=str, columns={"Organic carbonTotal (mg/L)":"TOC_mgL","Organic carbonDissolved (mg/L)":"DOC_mgL","SulfateTotal (mg/L)":"TS_mgL",
                                          "Total suspended solids (mg/L)":"TSS_mgL","Total dissolved solids (mg/L)":"TDS_mgL"})

# add time and corrected formatted date columns
df_loc['time']=1200
df_loc['date'] = df_loc.Date.dt.strftime("%Y%m%d")
df_loc=df_loc.drop(['MercuryTotal (ng/L)','MercuryDissolved (ng/L)','Depth','ResultComment','Latitude','Longitude'],axis=1)

# make sure dates are in the right order
df_loc = df_loc.sort_values(by=['Location','Date'])

# Determine range of dates to output
df_loc = df_loc.loc[lambda AA: AA.Date > startdate, :]
df_loc = df_loc.loc[lambda AA: AA.Date < enddate, :]

In [57]:
######################################
# Import hydrometric flow data and merge 
######################################

# Create empty dataframe to store data for all rivers
Q_all = pd.DataFrame()

# Read in relevant flow data and combine it
for X in file_name:
    Qname = Flow_name+X+'.csv'
    Q = pd.read_csv(Qname, skiprows=1)
    Q = Q.loc[lambda AA: AA.Date > startdate, :]
    Q = Q.loc[lambda BB: BB.PARAM == 1,:]
    Q['Q'] = Q['Value']*35.3
    Q = Q.drop(['PARAM','SYM','Value'],axis=1)
    Q_all = Q_all.append(Q)

# Rename rivers so that names are consistent with constituent dataset
Q_all = Q_all.rename(index=str, columns={" ID":"Location"})
Q_all = Q_all.set_index(['Location','Date'])
Q_all = Q_all.rename({"10MC002":'Peel River','10ED002':'Liard River',
              '10LA002':'Arctic Red River','07SB003':'Yellowknife River'})

# Reformat date
Q_all = Q_all.reset_index()
Q_all["Date"]=pd.to_datetime(Q_all.Date)

######################################################
# Create empty dataframe to store data for all rivers
New_all = pd.DataFrame()

i=0
# Read in relevant flow data and combine it
for X in file_name:
    fname1='/Users/staffan/Dropbox/Mackenzie Hg project/Supporting data/Hydrological_data/2018_Daily_'+X+'.xlsx'
    New = pd.read_excel(fname1, sheet_name='Sheet1')
    New['Q']=New['Mean (m3/s)']*35.3
    #subtract a year (2019-1) as the year is wrong in the input file
    New['Date']=New['Day (m-d)']- timedelta(days=365)
    #New['date'] = New['Day (m-d)'].dt.strftime("%Y%m%d")
    New = New.drop(['Max (m3/s)','Min (m3/s)','Median (m3/s)','Upper Quartile (m3/s)','Lower Quartile (m3/s)',
                    'Mean (m3/s)','Day (m-d)'], axis=1)#
    New['Location']=River_short[i]
    New = New[['Location','Date','Q']]
    New_all = New_all.append(New)
    i=i+1
    
Q_all = Q_all.append(New_all)

# Merge constituent data with hydrological flows
dfQ =pd.merge(df_loc,Q_all,how='left',left_on=['Location','Date'],right_on=['Location','Date'])
dfQ = dfQ.dropna(subset=['Q'])

dfQ.head(50)

,Location,Date,TOC_mgL,DOC_mgL,TS_mgL,TDS_mgL,TSS_mgL,THg_ugL,DHg_ugL,time,date,Q
0,Arctic Red River,2013-08-11,6.0,6.0,124.0,286.0,106.0,0.00858,0.00076,1200,20130811,7660.10
1,Arctic Red River,2013-09-08,5.9,5.7,133.0,312.0,206.0,0.01540,0.00128,1200,20130908,11154.80
2,Arctic Red River,2014-07-15,4.8,4.7,125.0,334.0,188.0,0.01540,0.00070,1200,20140715,9989.90
3,Arctic Red River,2014-08-12,7.1,7.2,117.0,268.0,242.0,0.01280,0.00080,1200,20140812,11613.70
4,Arctic Red River,2014-08-12,7.4,7.3,117.0,282.0,292.0,0.01330,0.00070,1200,20140812,11613.70
5,Arctic Red River,2014-08-12,7.2,7.1,118.0,288.0,188.0,0.01430,0.00070,1200,20140812,11613.70
6,Arctic Red River,2015-07-13,4.6,4.6,139.0,302.0,436.0,0.03810,0.00050,1200,20150713,11472.50
7,Arctic Red River,2015-08-12,4.9,5.1,150.0,319.0,105.0,0.00900,0.00090,1200,20150812,8648.50
8,Arctic Red River,2015-09-09,13.3,12.6,80.0,223.0,735.0,0.03350,0.00130,1200,20150909,20403.40
9,Arctic Red River,2016-07-13,6.6,6.5,103.0,281.0,464.0,0.03680,0.00080,1200,20160713,25274.80


In [58]:
############################################
# Create .inp files based on merged constituent and hydrological flow file
##############################################

i=0
# Loop over the different variables that we want to create input files for
for idd in River_short:
    for id in NWT_list:

        dfV = dfQ.loc[lambda BB: BB.Location == idd,:]
        dfV = dfV.loc[:,('date','time','Q',id)]

        # Save excel input as a textfile
        Oname = Temp_name+'/calib_temp_NWT_'+idd+id+'.txt'
        dfV.to_csv(Oname, sep='\t', index=False, header=False)

        # Import text file
        Tname = Temp_name+'/calib_temp_NWT_'+idd+id+'.txt'
        calib_file = open(Tname,'r')
        calib = calib_file.read()

        ###########################################
        # Import original calib.inp file
        days_file = open(path,'r')
        days = days_file.read()

        ############################################
        # Combine the header with the datafile
        days1 = days[1:106]
        days01 = idd
        days02 = days[116:121]
        days2 = id
        days3 = days[124:309]
        days4 = days1+days01+days02+days2+days3+calib
        #print(days)
        ############################################
        # Write new calib.inp file
        new_path = Out_name+'/calib_NWT_'+idd+"_"+id+'.inp'
        new_days = open(new_path,'w')

        new_days.write(days4)
        days_file.close()
        new_days.close()

    i=i+1

In [59]:
print(days4)

#####################################################################
#
#  LOADEST Calibration File
#
#  Yellowknife Riveriver TDS_mgLMarseilles, Illinois (Helsel & Hirsch, 2002)
#
#  Note: Sample dates (CDATE) were extracted from the decimal times
#        given by Helsel and Hirsch (2002).  Sample times (CTIME) are
20130704	1200	716.5899999999999	440.0
20130801	1200	674.23	46.0
20130828	1200	773.0699999999999	16.0
20140618	1200	695.41	32.0
20140618	1200	695.41	36.0
20140618	1200	695.41	44.0
20140814	1200	744.83	50.0
20140905	1200	773.0699999999999	40.0
20150717	1200	182.148	48.0
20150720	1200	177.20599999999996	5.0
20150814	1200	148.613	43.0
20150909	1200	138.72899999999998	40.0
20151007	1200	157.79099999999997	32.0
20160719	1200	1013.1099999999999	49.0
20160810	1200	875.4399999999999	40.0
20160812	1200	871.9099999999999	35.0
20160908	1200	624.81	20.0
20160930	1200	529.5	25.0
20180810	1200	1436.71	5.0
20180917	1200	1214.32	14.0

